In [1]:
import sys, os

# Fix path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
SRC_DIR      = os.path.join(PROJECT_ROOT, "src")

for p in [SRC_DIR, PROJECT_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"✅ SRC: {SRC_DIR}")
print(f"✅ ROOT: {PROJECT_ROOT}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load NASA dataset CSV
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "battery_data.csv")

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Loaded NASA dataset: {df.shape}")
else:
    print("❌ battery_data.csv not found!")

plt.style.use("dark_background")
sns.set_palette("Set2")
print("✅ Ready!")



✅ SRC: c:\Users\HP\ecocharge\src
✅ ROOT: c:\Users\HP\ecocharge
✅ Loaded NASA dataset: (636, 9)
✅ Ready!


In [3]:

import sys
sys.path.insert(0, '../src')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from preprocess import run_preprocessing_pipeline
from evaluate  import compute_metrics, plot_predictions, plot_soh_degradation_curve

plt.style.use('dark_background')
print("✅ Imports complete")

# %% [markdown]
# ## 2. Load Model & Test Data

# %%
model = tf.keras.models.load_model('../models/lstm_model.h5')
print("✅ Model loaded")
model.summary()

# %%
_, _, X_test, _, _, y_test = run_preprocessing_pipeline()
print(f"Test set: {X_test.shape}")

# %% [markdown]
# ## 3. Predictions

# %%
y_pred = model.predict(X_test, verbose=1).flatten()
print(f"\nPrediction range: {y_pred.min():.4f} – {y_pred.max():.4f}")
print(f"Actual range    : {y_test.min():.4f} – {y_test.max():.4f}")

# %% [markdown]
# ## 4. Metrics

# %%
metrics = compute_metrics(y_test, y_pred)

# Also display as a formatted table
metrics_df = pd.DataFrame({'Metric': list(metrics.keys()), 'Value': list(metrics.values())})
print("\n")
print(metrics_df.to_string(index=False))

# %%
# Save metrics
os.makedirs('../results', exist_ok=True)
with open('../results/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("✅ Metrics saved → ../results/metrics.json")

# %% [markdown]
# ## 5. Actual vs Predicted Plot

# %%
plot_predictions(y_test, y_pred)

# Interactive inline version
fig, ax = plt.subplots(figsize=(14, 5))
n = min(400, len(y_test))
ax.plot(y_test[:n], label='Actual Capacity',    color='deepskyblue',  linewidth=1.5)
ax.plot(y_pred[:n], label='Predicted Capacity', color='darkorange',   linewidth=1.5, linestyle='--')
ax.fill_between(range(n), y_test[:n], y_pred[:n], alpha=0.1, color='red', label='Error')
ax.set_title('Test Set — Actual vs Predicted Battery Capacity', fontsize=13, fontweight='bold')
ax.set_xlabel('Sample Index'); ax.set_ylabel('Capacity (Ah)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 6. SoH Degradation Curve

# %%
plot_soh_degradation_curve(y_test, y_pred)

# %% [markdown]
# ## 7. Error Analysis

# %%
residuals = y_test - y_pred
print(f"Residual stats:")
print(f"  Mean  : {residuals.mean():.6f}")
print(f"  Std   : {residuals.std():.6f}")
print(f"  Min   : {residuals.min():.6f}")
print(f"  Max   : {residuals.max():.6f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals over time
axes[0].plot(residuals[:300], color='mediumpurple', linewidth=0.8)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals Over Samples')
axes[0].set_xlabel('Sample'); axes[0].set_ylabel('Residual (Ah)')
axes[0].grid(alpha=0.3)

# Histogram
axes[1].hist(residuals, bins=60, color='mediumpurple', edgecolor='none', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Residuals Distribution')
axes[1].set_xlabel('Residual (Ah)'); axes[1].set_ylabel('Count')
axes[1].grid(alpha=0.3)

# Q-Q style: pred vs actual scatter
axes[2].scatter(y_test, y_pred, alpha=0.2, s=6, color='deepskyblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[2].plot(lims, lims, 'r--', linewidth=1.5)
axes[2].set_xlabel('Actual (Ah)'); axes[2].set_ylabel('Predicted (Ah)')
axes[2].set_title('Scatter: Pred vs Actual')
axes[2].grid(alpha=0.3)

plt.suptitle('Error Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/error_analysis.png', dpi=150)
plt.show()

# %% [markdown]
# ## 8. Lifecycle Recommendation Demo

# %%
from predict import capacity_to_soh, get_recommendation, estimate_rul

NOMINAL = 1.9
soh_series   = [(c / NOMINAL) * 100 for c in y_pred]
cycle_series = list(range(1, len(soh_series) + 1))

# Fix: only use valid indices
max_idx = len(soh_series) - 1
check_points = [int(max_idx * 0.25), int(max_idx * 0.5), 
                int(max_idx * 0.75), max_idx]

for cycle_idx in check_points:
    soh = soh_series[cycle_idx]
    rul = estimate_rul(soh_series[:cycle_idx+1], cycle_series[:cycle_idx+1])
    rec = get_recommendation(soh)
    print(f"Cycle {cycle_series[cycle_idx]:4d} | SoH: {soh:.1f}% | RUL: {rul:5d} cycles | {rec['action']}")

# %% [markdown]
# ## 9. Performance Summary Table

# %%
summary = pd.DataFrame({
    'Model':       ['LSTM (EcoCharge)'],
    'MAE (Ah)':   [metrics['MAE']],
    'RMSE (Ah)':  [metrics['RMSE']],
    'R² Score':   [metrics['R2']],
    'MAPE (%)':   [metrics['MAPE']],
})
print("\n" + "=" * 55)
print("  FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 55)
print(summary.to_string(index=False))

print("✅ Evaluation complete!")

✅ Imports complete
✅ Model loaded
Model: "EcoCharge_LSTM"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 battery_sequence (InputLay  [(None, 30, 7)]           0         
 er)                                                             
                                                                 
 lstm_1 (LSTM)               (None, 30, 32)            5120      
                                                                 
 dropout_1 (Dropout)         (None, 30, 32)            0         
                                                                 
 lstm_2 (LSTM)               (None, 16)                3136      
                                                                 
 dropout_2 (Dropout)         (None, 16)                0         
                                                                 
 dense_1 (Dense)             (None, 16)                272       
                  

c:\Users\HP\ecocharge\notebooks\../src\preprocess.py:59: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('battery_id', group_keys=False).apply(lambda g: g.ffill().bfill())


3/3 [==============================] - 1s 9ms/step

Prediction range: 1.4205 – 1.5757
Actual range    : 1.1852 – 1.8553

[evaluate] ── Test-Set Metrics ──────────────────────
  MAE   : 0.153018
  MSE   : 0.03598353
  RMSE  : 0.189693
  R2    : -0.321
  MAPE  : 10.1138
─────────────────────────────────────────────────────


Metric     Value
   MAE  0.153018
   MSE  0.035984
  RMSE  0.189693
    R2 -0.321000
  MAPE 10.113800
✅ Metrics saved → ../results/metrics.json
[evaluate] Saved → actual_vs_predicted.png
[evaluate] Saved → scatter_plot.png
[evaluate] Saved → residuals.png


C:\Users\HP\AppData\Local\Temp\ipykernel_6096\1997727901.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[evaluate] Saved → soh_curve.png
Residual stats:
  Mean  : 0.015191
  Std   : 0.189084
  Min   : -0.354763
  Max   : 0.400169
Cycle   20 | SoH: 79.1% | RUL:  9999 cycles | ⚠️  Repurpose
Cycle   39 | SoH: 78.1% | RUL:  9999 cycles | ⚠️  Repurpose
Cycle   58 | SoH: 80.4% | RUL:     0 cycles | ✅  Continue Use
Cycle   78 | SoH: 79.5% | RUL:     0 cycles | ⚠️  Repurpose

  FINAL MODEL PERFORMANCE SUMMARY
           Model  MAE (Ah)  RMSE (Ah)  R² Score  MAPE (%)
LSTM (EcoCharge)  0.153018   0.189693    -0.321   10.1138
✅ Evaluation complete!


C:\Users\HP\AppData\Local\Temp\ipykernel_6096\1997727901.py:119: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
